# 2D Swift-Hohenberg Equation
## Pattern Formation, Localized Structures, and Subcritical Dynamics

This notebook simulates the generalized 2D Swift-Hohenberg equation using the pseudo-spectral `PDESolver` framework, following the formulation from [VisualPDE](https://visualpde.com/nonlinear-physics/swift-hohenberg.html). This equation is a paradigmatic model for pattern formation, studying phenomena from Rayleigh-Bénard convection to nonlinear optics and fluid dynamics.

---

## 1. The Governing Equation

$$
\partial_t u = r u - (k_c^2 + \nabla^2)^2 u + a u^2 + b u^3 + c u^5
$$

* $r$ (Bifurcation Parameter): Controls the distance from the instability threshold. When $r > 0$, the uniform state $u=0$ becomes unstable.
* $k_c$ (Critical Wavenumber): Sets the preferred wavelength of patterns: $\lambda_c = 2\pi/k_c$.
* $a, b, c$ (Nonlinear Coefficients): Determine the nature of the bifurcation and pattern selection.
  * $c < 0$ (or $b < 0$ if $c=0$) is required for stability.
  * $a > 0, b < 0$: **Subcritical regime** supporting localized structures and multistability.

---

## 2. Reformulation for the Solver

We expand the linear operator in Fourier space, where $\nabla^2 \to -k^2 = -(\xi^2 + \eta^2)$:

$$
-(k_c^2 + \nabla^2)^2 \longrightarrow -(k_c^2 - k^2)^2
$$

The full linear symbol is:

$$
\text{Linear symbol:} \quad r - (k_c^2 - (\xi^2 + \eta^2))^2
$$

The equation in the solver's format:

$$
\partial_t u = \underbrace{u_{\text{op}}\!\left(r - (k_c^2 - k^2)^2\right)u} {\text{Linear band-pass filter}} + \underbrace{a u^2 + b u^3 + c u^5} {\text{Nonlinear terms}}
$$

---

## 3. Physical Phenomena

* **Supercritical Patterns** ($r > 0, a=0, b<0$): Spontaneous formation of stripes, hexagons, or labyrinthine patterns.
* **Subcritical Regime** ($r < 0, a > 0, b < 0$): Coexistence of stable homogeneous state ($u=0$) and stable patterned states, enabling **localized structures**.
* **Localized Solutions**: Spatially confined patterns that decay to $u=0$ away from the core — analogous to solitons in dissipative systems.
* **Symmetry Selection**: Different initial conditions can select patterns with different symmetries (D4, D6, D12).

We provide two examples:
1. **Standard pattern formation** from random noise
2. **Localized structures** in the subcritical regime

# Implementation
## 0. Imports 

In [ ]:
from solver import PDESolver, psiOp  # psiOp for real-valued fields
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML

## 1. Physical and simulation parameters 

In [ ]:
# ── Swift-Hohenberg Coefficients ──
# Example 1: Standard pattern formation (supercritical)
R = 0.2         # Bifurcation parameter (r > 0 triggers instability)
KC = 1.0        # Critical wavenumber (preferred wavelength λ = 2π/kc ≈ 6.28)
A = 0.0         # Quadratic nonlinearity (a=0 for standard SH)
B = -1.0        # Cubic nonlinearity (b<0 for stability)
C = 0.0         # Quintic nonlinearity (c=0 for standard SH)

# Uncomment for Example 2: Localized structures (subcritical regime)
# R = -0.25       # r < 0 (homogeneous state is stable)
# KC = 1.0
# A = 2.0         # a > 0 (subcritical)
# B = -1.5        # b < 0 (stabilizing)
# C = 0.0

# ── Grid and Time ─
Lx, Ly = 40.0, 40.0   # Large domain to fit multiple wavelengths
Nx, Ny = 256, 256     # High resolution for sharp interfaces

Lt, Nt = 100.0, 2000  # Long time for pattern coarsening
n_frames = 200

## 2. Grid setup 

In [ ]:
xs_1d = np.linspace(-Lx/2, Lx/2, Nx)
ys_1d = np.linspace(-Ly/2, Ly/2, Ny)

# CRITICAL FIX: psipy internally uses indexing='ij' (axis 0 = x, axis 1 = y).
xx, yy = np.meshgrid(xs_1d, ys_1d, indexing='ij')   # shape (Nx, Ny)

## 3. SymPy symbols and principal symbol 

In [ ]:
t, x, y   = sp.symbols('t x y', real=True)
xi, eta   = sp.symbols('xi eta', real=True)
u_func    = sp.Function('u')
u_field   = u_func(t, x, y)

# ── Linear symbol in Fourier space ──
# From: u/∂t = r·u - (kc² + ∇²)²u + ...
# Fourier: ∇² → -(ξ² + η²)
# So: r - (kc² - (ξ² + η²))²

k2 = xi**2 + eta**2
symbol_linear = R - (KC**2 - k2)**2

print('Principal symbol (linear part):')
print('  a(ξ, η) = ', symbol_linear)
print(f'\nPreferred wavelength: λ = {2*np.pi/KC:.2f}')
print(f'Linear growth rate at k=kc: {R}')

## 4. Swift-Hohenberg equation 

In [ ]:
# ∂u/∂t = psiOp(r - (kc² - k²)², u)  +  au² + bu³ + cu
#        ────────────────────────    ────────────────────
#        Linear band-pass filter     Nonlinear saturation

equation = sp.Eq(
    sp.diff(u_field, t),
    psiOp(symbol_linear, u_field) + A * u_field**2 + B * u_field**3 + C * u_field**5
)

print('Swift-Hohenberg Equation:')
print('  ∂u/∂t = psiOp(r - (kc² - k²)², u) + au² + bu³ + cu⁵')
print(f'\nParameters: r={R}, kc={KC}, a={A}, b={B}, c={C}')

## 5. Initial conditions 

In [ ]:
def initial_condition_patterns(xx, yy):
    """
    Example 1: Small random noise to trigger pattern formation.
    The linear instability amplifies modes near k=kc, leading to
    spontaneous symmetry breaking and pattern emergence.
    """
    np.random.seed(42)
    noise_amplitude = 0.01
    return noise_amplitude * np.random.randn(xx.shape[0], xx.shape[1])


def initial_condition_localized(xx, yy):
    """
    Example 2: Localized perturbation for subcritical regime.
    A sufficiently large localized bump can trigger a stable 
    localized structure that persists in the background u≈0.
    Based on D4-symmetric initial condition from VisualPDE.
    """
    # Localized Gaussian bump at center
    r2 = xx**2 + yy**2
    amplitude = 1.5  # Must be large enough to escape u=0 basin
    width = 3.0
    return amplitude * np.exp(-r2 / width**2)


# Choose initial condition based on regime
if R > 0 and A == 0:
    initial_condition_sh = initial_condition_patterns
    print("Using: Random noise for pattern formation")
else:
    initial_condition_sh = initial_condition_localized
    print("Using: Localized perturbation for subcritical structures")

## 6. Solver setup 

In [ ]:
solver = PDESolver(equation)

solver.setup(
    Lx=Lx, Ly=Ly,
    Nx=Nx, Ny=Ny,
    Lt=Lt, Nt=Nt,
    boundary_condition='periodic',  # Standard for pattern formation
    initial_condition=initial_condition_sh,
    n_frames=n_frames,
    plot=True,
)

## 7. Solve 

In [ ]:
frames = solver.solve()

## 8. Visualization 

In [ ]:
plt.rcParams['animation.embed_limit'] = 2**128

ani = solver.animate(
    component='real',
    overlay=None,
    mode='surface',      # 'surface' shows the pattern amplitude clearly
    physical=True
)

HTML(ani.to_jshtml())

In [ ]:
ani.save('swift_hohenberg_visualpde.mp4', writer='ffmpeg', fps=20, dpi=100)
print("✅ Saved to swift_hohenberg_visualpde.mp4")

print("\n" + "="*60)
print("TRY THESE PARAMETER COMBINATIONS:")
print("="*60)
print("\n1. Standard patterns (supercritical):")
print("   R = 0.2, KC = 1.0, A = 0.0, B = -1.0, C = 0.0")
print("   → Stripes, hexagons, labyrinthine patterns")
print("\n2. Localized structures (subcritical):")
print("   R = -0.25, KC = 1.0, A = 2.0, B = -1.5, C = 0.0")
print("   → Spatially confined patterns on u=0 background")
print("\n3. Hexagonal patterns (D6 symmetry):")
print("   Change initial condition to favor hexagonal symmetry")
print("="*60)